# Skenario 3 — Perbandingan Model 1 (Greedy Init) vs Model 2 (Pure COA / Random Init)
### Penjadwalan Mata Kuliah dengan Coati Optimization Algorithm (COA)

Notebook ini **tidak menjalankan eksperimen baru**. Ia **membaca hasil yang sudah tersimpan** dari kedua model — file `detail_per_run.csv` dan checkpoint konvergensi di folder `Model1` dan `Model2` — lalu membandingkannya untuk menjawab rumusan masalah Skenario 3: *apakah inisialisasi greedy (Model 1) menghasilkan solusi lebih baik dibanding pure COA dengan inisialisasi acak (Model 2)?*

**Prasyarat:** Notebook Model 1 dan Model 2 sudah dijalankan penuh, sehingga kedua folder berisi `detail_per_run.csv`.

**Yang dibandingkan (dan catatannya):**
- **Fitness & penalty akhir** — bisa dibandingkan, namun Model 2 memiliki constraint tambahan **H7 (dosen tidak berminat)** yang tidak ada di Model 1, sehingga penalti Model 2 dapat lebih tinggi karena sumber pelanggaran ekstra. Perbandingan tetap dilakukan, dengan catatan ini.
- **Execution time** — dapat dibandingkan langsung (adil).
- **Feasible rate** — dapat dibandingkan, dengan catatan definisi feasible Model 2 lebih ketat (H7 juga harus nol).
- **Constraint bersama (H1–H6, S1)** — perbandingan paling adil karena ada di kedua model.
- **Kurva konvergensi** — fitness terbaik vs iterasi untuk kedua model.

**Yang TIDAK dibandingkan:** `reduction_pct` (hanya ada di Model 1, karena Model 2 tak punya baseline greedy).

> Karena Model 1 dan Model 2 berbeda pada dua faktor (inisialisasi **dan** penanganan dosen), perbandingan ini bersifat *greedy-assisted COA vs pure COA*, bukan sekadar uji inisialisasi. Interpretasi hasil harus menyebutkan kedua faktor tersebut.

## 1. Imports & Konfigurasi Path

Memuat pustaka dan menetapkan lokasi folder hasil kedua model serta folder output Skenario 3. **Sesuaikan `BASE_DIR` bila lokasi Anda berbeda.**

In [ ]:
# Pustaka analisis & visualisasi
import os                          # susun path & cek keberadaan file
import pandas as pd               # baca CSV hasil & susun tabel perbandingan
import numpy as np                # operasi numerik
import matplotlib.pyplot as plt   # visualisasi perbandingan
from scipy import stats           # uji statistik (Wilcoxon) antar-model
from IPython.display import display

# Folder induk tempat kedua model menyimpan hasil
BASE_DIR = r"D:\Berkas UB\SKRIPSI\COATI\Coati_Optimization_University-Course-Schedulling\experiment\Final Eksperimen"

# Folder hasil masing-masing model
DIR_M1 = os.path.join(BASE_DIR, "Model1")   # hasil Model 1 (greedy init)
DIR_M2 = os.path.join(BASE_DIR, "Model2")   # hasil Model 2 (pure COA random init)

# Folder output Skenario 3 (dibuat bila belum ada)
OUT_DIR = os.path.join(BASE_DIR, "Skenario3_Perbandingan")
os.makedirs(OUT_DIR, exist_ok=True)

def out_path(fname):
    return os.path.join(OUT_DIR, fname)

print("Folder Model 1 :", DIR_M1)
print("Folder Model 2 :", DIR_M2)
print("Output Skenario 3:", OUT_DIR)

## 2. Membaca Hasil Kedua Model

Membaca `detail_per_run.csv` dari kedua folder. Ditambahkan kolom `model` sebagai penanda, lalu digabung untuk analisis. Bila salah satu file tidak ada, notebook memberi tahu agar model tersebut dijalankan dulu.

In [ ]:
# Path file detail per-run kedua model
csv_m1 = os.path.join(DIR_M1, "detail_per_run.csv")
csv_m2 = os.path.join(DIR_M2, "detail_per_run.csv")

# Pastikan kedua file ada
for path, nama in [(csv_m1, "Model 1"), (csv_m2, "Model 2")]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File hasil {nama} tidak ditemukan: {path}\n"
            f"Jalankan notebook {nama} sampai selesai terlebih dahulu.")

# Baca & beri penanda model
df_m1 = pd.read_csv(csv_m1); df_m1['model'] = 'Model 1 (Greedy)'
df_m2 = pd.read_csv(csv_m2); df_m2['model'] = 'Model 2 (Pure COA)'

# Gabung (kolom yang berbeda otomatis diisi NaN, mis. H7 hanya di Model 2, reduction hanya di Model 1)
df_all = pd.concat([df_m1, df_m2], ignore_index=True, sort=False).fillna(0)

print(f"Model 1: {len(df_m1)} run")
print(f"Model 2: {len(df_m2)} run")
print(f"Kolom gabungan: {list(df_all.columns)}")
display(df_all.head())

## 3. Tabel Perbandingan Ringkas (per kombinasi parameter)

Merata-ratakan metrik utama per kombinasi `(population, max_iter, slot_interval)` untuk tiap model, lalu menyandingkannya berdampingan. Ini tabel inti Skenario 3.

In [ ]:
# Agregasi per (model, kombinasi parameter)
grp_cols = ['model', 'population', 'max_iter', 'slot_interval']
agg = (df_all.groupby(grp_cols)
       .agg(mean_fitness=('final_fitness', 'mean'),
            std_fitness =('final_fitness', 'std'),
            best_fitness=('final_fitness', 'max'),
            mean_penalty=('final_penalty', 'mean'),
            mean_exec_time=('execution_time', 'mean'),
            feasible_rate=('is_feasible', lambda s: (s==True).mean()*100 if s.dtype==bool else (s.astype(str)=='True').mean()*100))
       .reset_index())

# Simpan tabel perbandingan
tabel_csv = out_path("skenario3_tabel_perbandingan.csv")
agg.to_csv(tabel_csv, index=False)
print("Tabel perbandingan tersimpan:", tabel_csv)
display(agg)

## 4. Perbandingan Keseluruhan (rata-rata semua run)

Ringkasan satu baris per model: rata-rata fitness, penalty, execution time, dan feasible rate lintas seluruh run. Memberi gambaran cepat model mana yang unggul secara umum.

In [ ]:
# Fungsi bantu feasible rate (kolom is_feasible bisa bool atau string)
def feas_rate(s):
    if s.dtype == bool:
        return s.mean()*100
    return (s.astype(str)=='True').mean()*100

# Ringkasan per model
overall = df_all.groupby('model').agg(
    n_run=('final_fitness', 'size'),
    mean_fitness=('final_fitness', 'mean'),
    std_fitness=('final_fitness', 'std'),
    best_fitness=('final_fitness', 'max'),
    mean_penalty=('final_penalty', 'mean'),
    min_penalty=('final_penalty', 'min'),
    mean_exec_time=('execution_time', 'mean'),
).reset_index()
overall['feasible_rate_%'] = df_all.groupby('model')['is_feasible'].apply(feas_rate).values

overall_csv = out_path("skenario3_ringkasan_keseluruhan.csv")
overall.to_csv(overall_csv, index=False)
print("Ringkasan keseluruhan tersimpan:", overall_csv)
display(overall)

## 5. Visualisasi 1 — Distribusi Fitness Akhir per Model

Boxplot fitness akhir kedua model. Menunjukkan model mana yang cenderung menghasilkan solusi lebih baik dan lebih konsisten (kotak lebih tinggi & sempit = lebih baik & stabil).

In [ ]:
plt.figure(figsize=(8, 5))
data = [df_m1['final_fitness'].values, df_m2['final_fitness'].values]
plt.boxplot(data, labels=['Model 1\n(Greedy)', 'Model 2\n(Pure COA)'])
plt.ylabel("Fitness akhir (besar = lebih baik)")
plt.title("Perbandingan Distribusi Fitness Akhir")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path("skenario3_boxplot_fitness.png"), dpi=150)
plt.show()

## 6. Visualisasi 2 — Perbandingan Berpasangan per Konfigurasi

Untuk tiap kombinasi parameter yang sama, sandingkan mean fitness Model 1 vs Model 2. Garis penghubung menunjukkan arah keunggulan tiap konfigurasi (naik/turun antar model).

In [ ]:
# Rata-rata fitness per (kombinasi) tiap model, lalu sandingkan
piv = (df_all.groupby(['population','max_iter','slot_interval','model'])['final_fitness']
       .mean().reset_index())
# Bentuk lebar: satu baris per kombinasi, kolom per model
wide = piv.pivot_table(index=['population','max_iter','slot_interval'],
                       columns='model', values='final_fitness').reset_index()

col_m1 = 'Model 1 (Greedy)'
col_m2 = 'Model 2 (Pure COA)'

plt.figure(figsize=(9, 5))
for _, r in wide.iterrows():
    # garis penghubung tiap konfigurasi
    plt.plot([0, 1], [r[col_m1], r[col_m2]], color='gray', alpha=0.5, marker='o')
plt.xticks([0, 1], ['Model 1 (Greedy)', 'Model 2 (Pure COA)'])
plt.ylabel("Mean fitness per konfigurasi")
plt.title("Perbandingan Berpasangan per Konfigurasi Parameter")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path("skenario3_paired_fitness.png"), dpi=150)
plt.show()

## 7. Visualisasi 3 — Perbandingan Execution Time

Boxplot waktu komputasi kedua model. Perbandingan ini adil karena tidak dipengaruhi perbedaan constraint.

In [ ]:
plt.figure(figsize=(8, 5))
data = [df_m1['execution_time'].values, df_m2['execution_time'].values]
plt.boxplot(data, labels=['Model 1\n(Greedy)', 'Model 2\n(Pure COA)'])
plt.ylabel("Execution time (detik)")
plt.title("Perbandingan Waktu Komputasi")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path("skenario3_boxplot_exectime.png"), dpi=150)
plt.show()

## 8. Visualisasi 4 — Constraint Bersama (H1–H6, S1)

Perbandingan **paling adil**: rata-rata pelanggaran tiap constraint yang ada di **kedua** model. H7 (khusus Model 2) tidak disertakan di sini karena tak punya padanan di Model 1.

In [ ]:
# Constraint yang ada di KEDUA model (penomoran berurutan)
shared = ['H1_room_conflict', 'H2_lecturer_conflict', 'H3_lecturer_overload',
          'H4_room_type_mismatch', 'H5_lunch_break', 'H6_class_completeness',
          'S1_floor_movement']
# Pakai hanya yang benar-benar ada di kedua dataframe
shared = [c for c in shared if c in df_m1.columns and c in df_m2.columns]

# Rata-rata residual tiap constraint per model
res_m1 = [df_m1[c].mean() for c in shared]
res_m2 = [df_m2[c].mean() for c in shared]

x = np.arange(len(shared)); w = 0.35
plt.figure(figsize=(11, 5))
plt.bar(x - w/2, res_m1, w, label='Model 1 (Greedy)')
plt.bar(x + w/2, res_m2, w, label='Model 2 (Pure COA)')
plt.xticks(x, [c.replace('_', '\n') for c in shared], fontsize=8)
plt.ylabel("Rata-rata pelanggaran tersisa")
plt.title("Perbandingan Residual Constraint Bersama (H1–H6, S1)")
plt.legend(); plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(out_path("skenario3_constraint_bersama.png"), dpi=150)
plt.show()

## 9. Visualisasi 5 — Kurva Konvergensi Terbaik Kedua Model

Membaca checkpoint konvergensi run terbaik masing-masing model dan menggambar penurunan penalty vs iterasi berdampingan. Menunjukkan model mana yang konvergen lebih cepat / ke titik lebih baik.

In [ ]:
def kurva_terbaik(df_model, folder, label, warna):
    # Cari run dgn fitness akhir tertinggi
    r = df_model.loc[df_model['final_fitness'].idxmax()]
    fpath = os.path.join(folder,
        f"konvergensi_pop{int(r['population'])}_iter{int(r['max_iter'])}"
        f"_int{int(r['slot_interval'])}_seed{int(r['seed'])}.csv")
    if os.path.exists(fpath):
        cv = pd.read_csv(fpath)
        plt.plot(cv['iterasi'], cv['penalty'], label=label, color=warna)
        return f"{label}: pop={int(r['population'])}, iter={int(r['max_iter'])}, int={int(r['slot_interval'])}"
    return f"{label}: file konvergensi tak ditemukan"

plt.figure(figsize=(10, 5))
info1 = kurva_terbaik(df_m1, DIR_M1, 'Model 1 (Greedy)', 'C0')
info2 = kurva_terbaik(df_m2, DIR_M2, 'Model 2 (Pure COA)', 'C1')
plt.xlabel("Iterasi"); plt.ylabel("Penalty (kecil = lebih baik)")
plt.title("Kurva Konvergensi Konfigurasi Terbaik — Model 1 vs Model 2")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path("skenario3_konvergensi.png"), dpi=150)
plt.show()
print(info1); print(info2)

## 10. Uji Statistik (Wilcoxon Signed-Rank)

Menguji apakah perbedaan fitness kedua model **signifikan secara statistik**. Karena tiap konfigurasi dijalankan pada kedua model dengan parameter sama, digunakan uji berpasangan (Wilcoxon signed-rank) pada mean fitness per konfigurasi. p-value < 0,05 berarti perbedaan signifikan.

In [ ]:
# Ambil mean fitness per konfigurasi untuk kedua model (berpasangan)
wide_test = wide.dropna(subset=[col_m1, col_m2])

if len(wide_test) >= 5:
    stat, pval = stats.wilcoxon(wide_test[col_m1], wide_test[col_m2])
    print(f"Uji Wilcoxon Signed-Rank (n={len(wide_test)} konfigurasi):")
    print(f"  statistik = {stat:.4f}")
    print(f"  p-value   = {pval:.6f}")
    if pval < 0.05:
        menang = "Model 1 (Greedy)" if wide_test[col_m1].mean() > wide_test[col_m2].mean() else "Model 2 (Pure COA)"
        print(f"  -> Perbedaan SIGNIFIKAN (p<0.05). Rata-rata lebih tinggi: {menang}")
    else:
        print("  -> Perbedaan TIDAK signifikan (p>=0.05).")
else:
    print(f"Jumlah konfigurasi berpasangan ({len(wide_test)}) < 5; uji Wilcoxon kurang bermakna. "
          "Tambah konfigurasi/seed untuk uji yang lebih kuat.")

## 11. Laporan Ringkas Skenario 3

Menuliskan kesimpulan naratif perbandingan ke file teks, dengan catatan penting soal ketidaksetaraan (H7 & penanganan dosen).

In [ ]:
lines = []
lines.append("=== SKENARIO 3: PERBANDINGAN MODEL 1 (GREEDY) vs MODEL 2 (PURE COA) ===\n")
for _, r in overall.iterrows():
    lines.append(f"{r['model']}:")
    lines.append(f"  Mean fitness   : {r['mean_fitness']:.8f} (std {r['std_fitness']:.8f})")
    lines.append(f"  Best fitness   : {r['best_fitness']:.8f}")
    lines.append(f"  Mean penalty   : {r['mean_penalty']:.1f}  (min {r['min_penalty']:.0f})")
    lines.append(f"  Mean exec time : {r['mean_exec_time']:.2f} s")
    lines.append(f"  Feasible rate  : {r['feasible_rate_%']:.1f}%\n")

# Tentukan model unggul berdasar mean fitness
m1_fit = overall.loc[overall['model']=='Model 1 (Greedy)', 'mean_fitness'].values[0]
m2_fit = overall.loc[overall['model']=='Model 2 (Pure COA)', 'mean_fitness'].values[0]
unggul = "Model 1 (Greedy)" if m1_fit > m2_fit else "Model 2 (Pure COA)"
lines.append(f"KESIMPULAN: rata-rata fitness lebih tinggi pada {unggul}.")
lines.append("\nCATATAN PENTING:")
lines.append("- Model 2 memiliki constraint tambahan H7 (dosen tidak berminat) yang tidak ada di Model 1,")
lines.append("  sehingga penalti Model 2 dapat lebih tinggi karena sumber pelanggaran ekstra.")
lines.append("- Perbedaan kedua model mencakup DUA faktor: strategi inisialisasi (greedy vs acak)")
lines.append("  DAN penanganan dosen (fix vs variabel COA). Interpretasi harus menyebut keduanya.")

report = out_path("skenario3_laporan.txt")
with open(report, 'w') as f:
    f.write("\n".join(lines))
print("\n".join(lines))
print("\nLaporan tersimpan:", report)